# DATA209 — Advanced Exploratory Data Analysis
# Practical P19-20 · Outlier detection and treatment

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 10 · Module 3 · CO3

---

**Objective.** Detect outliers by several methods, decide per variable whether to treat, keep or investigate, and measure the impact of that decision.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 and P15-16 — the cleaned dataset, plus the scaler import used below.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# --- from P15-16
df_clean = df.drop_duplicates().reset_index(drop=True)
for c in ["VisitorType", "Month"]:
    df_clean[c] = df_clean[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

from sklearn.preprocessing import StandardScaler

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P19-20 — Outlier detection and treatment

### Detect outliers

Three univariate rules and one multivariate method. They disagree, and the disagreement is
informative — a point flagged by every method deserves investigation; a point flagged by one
may be an artefact of that rule's assumptions.

In [ ]:
# ---- Detection rules ----------------------------------------------------
def iqr_flags(s, k=1.5):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

def z_flags(s, t=3.0):
    sd = s.std()
    return (s - s.mean()).abs() / sd > t if sd else pd.Series(False, index=s.index)

def mad_flags(s, t=3.5):
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0:                       # fall back when more than half the values are identical
        mad = (s - med).abs().mean() or 1.0
        return 0.7979 * (s - med).abs() / mad > t
    return 0.6745 * (s - med).abs() / mad > t

check = ["Administrative_Duration", "Informational_Duration", "ProductRelated",
         "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues"]

detect = pd.DataFrame({
    "IQR_%"      : {c: iqr_flags(df_clean[c]).mean() * 100 for c in check},
    "z_score_%"  : {c: z_flags(df_clean[c]).mean() * 100 for c in check},
    "mod_z_%"    : {c: mad_flags(df_clean[c]).mean() * 100 for c in check},
    "skew"       : {c: df_clean[c].skew() for c in check},
})
print(detect.round(2).to_string())

print("\nInterpretation")
print("- The three rules disagree because each assumes a different shape.")
print("- The IQR fence flags the tail of a skewed distribution by construction, so a high")
print("  IQR_% on a high-skew column is expected — it is not evidence of bad data.")
print("- z-score under-flags here: the extreme values inflate the standard deviation that")
print("  the rule itself depends on. This is masking.")

In [ ]:
# ---- Multivariate detection ---------------------------------------------
from sklearn.ensemble import IsolationForest

mv_cols = ["Administrative", "Informational", "ProductRelated",
           "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues"]

iso = IsolationForest(contamination=0.02, random_state=RANDOM_STATE, n_estimators=200)
iso_flag = iso.fit_predict(StandardScaler().fit_transform(df_clean[mv_cols])) == -1

uni_any = np.zeros(len(df_clean), dtype=bool)
for c in mv_cols:
    uni_any |= iqr_flags(df_clean[c]).values

overlap = pd.crosstab(pd.Series(uni_any, name="flagged by any IQR rule"),
                      pd.Series(iso_flag, name="flagged by IsolationForest"))
print(overlap.to_string())

only_mv = int((iso_flag & ~uni_any).sum())
print(f"\nRows flagged ONLY by the multivariate method: {only_mv}")
print("These are ordinary on every single axis but implausible in combination —")
print("for example a very long session that viewed almost no pages.")

print("\nProfile of the multivariate-only outliers (medians):")
print(pd.DataFrame({
    "multivariate-only": df_clean.loc[iso_flag & ~uni_any, mv_cols].median(),
    "everyone else"    : df_clean.loc[~(iso_flag & ~uni_any), mv_cols].median(),
}).round(2).to_string())

### Treatment — and comparing before/after impact

Four options: **investigate, remove, treat, keep.** Never delete silently. The question that
settles most cases is: *does the decision actually change the answer?*

In [ ]:
# ---- Does it change the conclusion? ------------------------------------
def impact(frame, keep_mask, label):
    sub = frame[keep_mask]
    return {
        "scenario"       : label,
        "rows"           : len(sub),
        "PageValues_mean": sub["PageValues"].mean(),
        "PRD_mean"       : sub["ProductRelated_Duration"].mean(),
        "PRD_median"     : sub["ProductRelated_Duration"].median(),
        "PRD_std"        : sub["ProductRelated_Duration"].std(),
        "corr_PV_target" : sub["PageValues"].corr(sub[TARGET].astype(int)),
        "conversion_%"   : sub[TARGET].mean() * 100,
    }

rows = [
    impact(df_clean, np.ones(len(df_clean), dtype=bool), "all rows"),
    impact(df_clean, ~uni_any,  "drop univariate IQR flags"),
    impact(df_clean, ~iso_flag, "drop IsolationForest flags"),
]
comparison = pd.DataFrame(rows).set_index("scenario")
print(comparison.round(3).to_string())

base = comparison.loc["all rows"]
delta = ((comparison - base) / base * 100).drop(index="all rows")
print("\nPercentage change against 'all rows'")
print(delta.round(2).to_string())

print("\nInterpretation")
print("- Dropping the IQR flags removes a large share of rows and moves the correlation")
print("  materially. That is not cleaning — it is changing the dataset to suit the method.")
print("- The conversion rate barely moves, which tells you the outliers are not driving")
print("  the headline finding.")

In [ ]:
# ---- Treatment options compared ----------------------------------------
col = "ProductRelated_Duration"
s   = df_clean[col]

lo, hi   = s.quantile(0.01), s.quantile(0.99)
winsor   = s.clip(lo, hi)
logged   = np.log1p(s)
q1, q3   = s.quantile(.25), s.quantile(.75)
removed  = s[~iqr_flags(s)]

treat = pd.DataFrame({
    "treatment": ["none", "winsorise 1-99%", "log1p", "remove IQR flags"],
    "n"        : [len(s), len(winsor), len(logged), len(removed)],
    "mean"     : [s.mean(), winsor.mean(), logged.mean(), removed.mean()],
    "median"   : [s.median(), winsor.median(), logged.median(), removed.median()],
    "std"      : [s.std(), winsor.std(), logged.std(), removed.std()],
    "skew"     : [s.skew(), winsor.skew(), logged.skew(), removed.skew()],
    "IQR_out_%": [iqr_flags(s).mean()*100, iqr_flags(winsor).mean()*100,
                  iqr_flags(logged).mean()*100, iqr_flags(removed).mean()*100],
}).set_index("treatment")
print(treat.round(3).to_string())

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, (name, data) in zip(axes, [("none", s), ("winsorised", winsor),
                                   ("log1p", logged), ("IQR removed", removed)]):
    sns.histplot(data, bins=50, ax=ax, color="#A6752C")
    ax.set_title(f"{name} — skew {data.skew():.2f}"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

print("\nlog1p reduces skew dramatically while keeping every row. For a right-skewed")
print("positive variable it is almost always the better first move than deletion.")

In [ ]:
# ---- Per-variable decision table ---------------------------------------
decisions = pd.DataFrame([
    dict(variable="ProductRelated_Duration", flags=f"{iqr_flags(df_clean['ProductRelated_Duration']).sum():,}",
         decision="Transform (log1p)",
         reason="Right-skewed by nature; long sessions are genuine behaviour, not errors"),
    dict(variable="PageValues", flags=f"{iqr_flags(df_clean['PageValues']).sum():,}",
         decision="Keep",
         reason="The high values ARE the signal — they mark checkout-path sessions"),
    dict(variable="BounceRates", flags=f"{iqr_flags(df_clean['BounceRates']).sum():,}",
         decision="Keep",
         reason="Bounded in [0,1] and valid throughout; the spike at 0 is real behaviour"),
    dict(variable="Administrative_Duration", flags=f"{iqr_flags(df_clean['Administrative_Duration']).sum():,}",
         decision="Winsorise 99th",
         reason="Extreme tail is plausibly idle-time logging, not engagement"),
    dict(variable="multivariate-only rows", flags=str(only_mv),
         decision="Investigate",
         reason="Implausible combinations; check against the source before deciding"),
])
print(decisions.to_string(index=False))
print("\nWhatever you choose, the count and the justification go into the cleaning log.")

### Deliverable — P19-20

A notebook with the **per-variable decision table**, the before/after impact comparison, and
distribution plots for every variable you treated.